# Generación y comparación de atribuciones con XAI-metrics

Este notebook adapta el flujo de `prueba/main.py` a un ejemplo pequeño y reproducible. A partir de un mismo modelo **Isolation Forest**:

1. Genera atribuciones locales con **LIME** y **SHAP**.
2. Evalúa ambos métodos con las métricas `Complexity` y `Sparseness`.
3. Construye un reporte final que permite compararlos.

## 1. Preparar el entorno

El paquete debe estar instalado en modo editable con `pip install -e .`. También añadimos `prueba/` al path para reutilizar las funciones de generación de atribuciones y cargar el modelo de ejemplo.

In [ ]:
import sys
from pathlib import Path

import cloudpickle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from IPython.display import display
from lime.lime_tabular import LimeTabularExplainer

ROOT = Path.cwd()
if not (ROOT / "xai_metrics").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "prueba"))

from xai_metrics.base import MetricContext
from xai_metrics.reporting import build_reports
from xai_metrics.runner import run_evaluation

## 2. Definir los métodos de atribución

Para que el ejemplo sea autocontenido, definimos aquí las funciones locales `usar_lime` y `usar_shap_local`. Ambas devuelven una matriz con una fila por observación y una columna por variable.

In [ ]:
def usar_lime(clf, X_train, X_test, observaciones):
    """Genera atribuciones LIME locales para las observaciones indicadas."""
    feature_names = list(X_train.columns)
    explainer = LimeTabularExplainer(
        X_train.to_numpy(),
        feature_names=feature_names,
        mode="classification",
        random_state=42,
    )

    attributions = []
    for observation in observaciones:
        explanation = explainer.explain_instance(
            data_row=X_test.loc[observation].to_numpy(),
            predict_fn=clf.predict_proba,
            num_features=len(feature_names),
        )

        weights = np.zeros(len(feature_names))
        for feature_index, weight in explanation.as_map()[1]:
            weights[feature_index] = float(weight)
        attributions.append(weights)

    return np.asarray(attributions)


def usar_shap_local(clf, X_train, X_test, observaciones):
    """Genera atribuciones SHAP locales para el score de anomalía."""
    def predict_anomaly_score(values):
        return clf.decision_function(values)

    explainer = shap.Explainer(predict_anomaly_score, X_train)
    shap_values = explainer(X_test.loc[observaciones])

    return np.abs(shap_values.values)

## 3. Cargar el modelo y los datos

El modelo guardado contiene un wrapper compatible con las métricas de la librería. Para generar las atribuciones, LIME y SHAP utilizan el modelo PyOD original disponible en `wrapped_model.model`.

In [ ]:
X_train = pd.read_csv(ROOT / "prueba/data/hydraulic/X_train_norm.csv", index_col=0)
X_test = pd.read_csv(ROOT / "prueba/data/hydraulic/X_test_train_norm.csv", index_col=0)
y_test = pd.read_csv(ROOT / "prueba/data/hydraulic/y_test_train.csv", index_col=0).iloc[:, 0]

model_path = ROOT / "prueba/results/models/hydraulic/IForest/hydraulic_IForest_seed_0.pkl"
with model_path.open("rb") as model_file:
    wrapped_model = cloudpickle.load(model_file)

observations = [9, 5, 249, 252, 255, 685, 403, 420]
X_observations = X_test.loc[observations]

print(f"Datos de entrenamiento: {X_train.shape}")
print(f"Observaciones que se explicarán: {observations}")
X_observations

## 4. Generar atribuciones LIME y SHAP

Los dos métodos explican exactamente las mismas observaciones. Se usa una muestra reproducible de 100 filas como conjunto de referencia para acelerar el ejemplo.

In [ ]:
X_background = X_train.sample(n=100, random_state=42)
pyod_model = wrapped_model.model

lime_values = usar_lime(
    clf=pyod_model,
    X_train=X_background,
    X_test=X_test,
    observaciones=observations,
)

shap_values = usar_shap_local(
    clf=pyod_model,
    X_train=X_background,
    X_test=X_test,
    observaciones=observations,
)

lime_attributions = pd.DataFrame(lime_values, index=observations, columns=X_test.columns)
shap_attributions = pd.DataFrame(shap_values, index=observations, columns=X_test.columns)

lime_attributions.index.name = "observation"
shap_attributions.index.name = "observation"

print("Atribuciones LIME")
display(lime_attributions)
print("Atribuciones SHAP")
display(shap_attributions)

## 5. Comparar visualmente las atribuciones

La siguiente gráfica compara la importancia absoluta media de cada variable. Esta comparación describe las atribuciones; la evaluación cuantitativa con XAI-metrics se realiza en el siguiente paso.

In [ ]:
mean_absolute_attributions = pd.DataFrame({
    "LIME": lime_attributions.abs().mean(),
    "SHAP": shap_attributions.abs().mean(),
})

ax = mean_absolute_attributions.plot.bar(figsize=(9, 4))
ax.set_title("Importancia absoluta media por método")
ax.set_xlabel("Variable")
ax.set_ylabel("Atribución absoluta media")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Evaluar ambos métodos con XAI-metrics

Cada matriz de atribuciones se introduce en un `MetricContext`. Ambos contextos comparten modelo, datos, etiquetas y observaciones, por lo que la comparación se realiza en las mismas condiciones.

- `Complexity`: valores menores indican explicaciones menos complejas.
- `Sparseness`: valores mayores indican explicaciones más concentradas en pocas variables.

In [ ]:
metrics_config = {
    "metrics": [
        {"name": "Complexity", "params": {"normalise": True}},
        {"name": "Sparseness", "params": {"normalise": True}},
    ]
}


def evaluate_attributions(method_name, attributions):
    context = MetricContext(
        model=wrapped_model,
        X_test=X_test,
        y_test=y_test,
        observations=observations,
        attributions=attributions.to_numpy(),
        device="cpu",
    )

    return run_evaluation(
        context=context,
        metadata={
            "dataset_name": "hydraulic",
            "model_name": "IForest",
            "xai_method_name": method_name,
        },
        config=metrics_config,
        report_output_dir=None,
    )


lime_results = evaluate_attributions("LIME", lime_attributions)
shap_results = evaluate_attributions("SHAP", shap_attributions)

## 7. Reporte comparativo final

`build_reports` combina los resultados de ambos contextos. Las filas son métricas y las columnas son los métodos XAI comparados.

In [ ]:
combined_reports = build_reports(
    lime_results["contexts"] + shap_results["contexts"]
)

comparison = combined_reports["hydraulic"]["IForest"]
comparison

In [ ]:
ax = comparison.plot.bar(figsize=(8, 4))
ax.set_title("Comparación de métricas XAI")
ax.set_xlabel("Métrica")
ax.set_ylabel("Valor medio")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

Para evaluar otras métricas, basta con añadirlas a `metrics_config`. Algunas métricas de robustez o fidelidad también necesitan recibir una función capaz de regenerar explicaciones durante la evaluación.